In [4]:
# Cell 1: Install all required packages
# These are the GPU-specific packages that we couldn't install on Mac

!pip install -q transformers==4.52.3
!pip install -q peft==0.15.2
!pip install -q accelerate==1.7.0
!pip install -q bitsandbytes==0.45.5
!pip install -q trl==0.18.1
!pip install -q datasets==3.6.0

print("✅ All packages installed!")


✅ All packages installed!


In [5]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9  # Convert to GB
    print(f"✅ GPU detected: {gpu_name}")
    print(f"   VRAM: {gpu_memory:.1f} GB")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

✅ GPU detected: Tesla T4
   VRAM: 15.6 GB


In [6]:
# Cell 3: Load our preprocessed data files
from datasets import load_dataset

# Load JSONL files into HuggingFace Dataset objects
dataset = load_dataset("json", data_files={
    "train": "train.jsonl",
    "validation": "val.jsonl",
    "test": "test.jsonl"
})

print("📊 Dataset loaded:")
print(dataset)

print(f"\n📝 Sample training example:")
print(dataset["train"][0]["text"][:500])


Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

📊 Dataset loaded:
DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 897
    })
    validation: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 105
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 106
    })
})

📝 Sample training example:
### Instruction:
You are a financial sentiment analyst. Classify the sentiment of the following earnings call transcript segment. Respond with exactly one word: positive, negative, or neutral.

### Input:
I just wanted to follow up on AWS for a moment. You outlined the generative AI stack, which I think is -- which is very clear. So I'm just curious maybe how you're going to market within the application layer given sort of the competitive dynamics of that. And then maybe expand, if you could, A


In [7]:
# Cell 4: Load the base model with 4-bit quantization (QLoRA)

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# ─── What model are we fine-tuning? ───────────────────────────────
# Qwen2.5-7B: A modern, open-source 7 billion parameter model
# It's free to use, no login required, and works great for classification
MODEL_NAME = "Qwen/Qwen2.5-7B"

# ─── 4-bit Quantization Config ───────────────────────────────────
# This is the "Q" in QLoRA
# Instead of storing each model weight as a 16-bit number (2 bytes),
# we store it as a 4-bit number (0.5 bytes)
# This reduces memory from ~14GB to ~3.5GB — that's how it fits on a T4!

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # Enable 4-bit quantization
    bnb_4bit_quant_type="nf4",            # NormalFloat4 — best quality 4-bit format
    bnb_4bit_compute_dtype=torch.float16, # Do math in float16 (fast on T4)
    bnb_4bit_use_double_quant=True,       # Quantize the quantization constants too (saves more memory)
)

print(f"📦 Loading model: {MODEL_NAME}")
print("   This downloads ~4GB and may take 3-5 minutes on first run...")

# ─── Load the tokenizer ──────────────────────────────────────────
# The tokenizer converts text → numbers (tokens) and back
# "Hello world" → [15496, 995] → "Hello world"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set padding token (some models don't have one by default)
# We need this for batch training — making all examples the same length
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Use end-of-sentence token as padding

# ─── Load the model ──────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,  # Apply 4-bit quantization
    device_map="auto",                         # Automatically put model on GPU
)

# Disable caching during training (saves memory)
model.config.use_cache = False

print(f"\n✅ Model loaded successfully!")
print(f"   Model size: {model.num_parameters() / 1e9:.1f}B parameters")
print(f"   Memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")


📦 Loading model: Qwen/Qwen2.5-7B
   This downloads ~4GB and may take 3-5 minutes on first run...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/3.95G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]


✅ Model loaded successfully!
   Model size: 7.6B parameters
   Memory used: 5.6 GB


In [8]:
# Cell 5: Configure LoRA (Low-Rank Adaptation)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# ─── Prepare the model for training ──────────────────────────────
# After loading in 4-bit, we need to prepare it for training
# This enables gradient computation on the quantized model
model = prepare_model_for_kbit_training(model)

# ─── LoRA Configuration ──────────────────────────────────────────
# LoRA adds tiny trainable layers on top of the frozen base model
# Think of it like putting sticky notes on a textbook — you don't
# rewrite the textbook, you just add notes where needed

lora_config = LoraConfig(
    r=16,                          # Rank of the adapter — higher = more capacity but slower
                                   # 16 is a good balance. Range: 4-64

    lora_alpha=32,                 # Scaling factor — controls how much the adapter affects output
                                   # Rule of thumb: set to 2x the rank

    lora_dropout=0.05,             # Dropout — randomly disables 5% of adapter weights during training
                                   # Prevents overfitting (model memorizing instead of learning)

    target_modules=[               # Which layers to add adapters to
        "q_proj",                  # Query projection (attention mechanism)
        "k_proj",                  # Key projection (attention mechanism)
        "v_proj",                  # Value projection (attention mechanism)
        "o_proj",                  # Output projection (attention mechanism)
        "gate_proj",               # Gate projection (feed-forward network)
        "up_proj",                 # Up projection (feed-forward network)
        "down_proj",               # Down projection (feed-forward network)
    ],

    bias="none",                   # Don't train bias terms (saves memory)
    task_type="CAUSAL_LM",         # We're fine-tuning a causal language model
)

# ─── Apply LoRA to the model ─────────────────────────────────────
model = get_peft_model(model, lora_config)

# Show how many parameters we're actually training
trainable_params, total_params = model.get_nb_trainable_parameters()
print(f"📊 Parameter Summary:")
print(f"   Total parameters:     {total_params:>12,}")
print(f"   Trainable parameters: {trainable_params:>12,}")
print(f"   Trainable %:          {100 * trainable_params / total_params:.2f}%")


📊 Parameter Summary:
   Total parameters:     7,655,986,688
   Trainable parameters:   40,370,176
   Trainable %:          0.53%


In [9]:
# Cell 6: Configure the training process

from trl import SFTTrainer
from transformers import TrainingArguments

# ─── Training Arguments ──────────────────────────────────────────
# These control HOW the model trains — learning speed, batch size, etc.

training_args = TrainingArguments(
    output_dir="./results",                # Where to save checkpoints during training

    # ── How many times to go through the data ──
    num_train_epochs=3,                    # Train for 3 full passes through the data
                                           # More epochs = model sees data more times
                                           # Too many = overfitting (memorization)

    # ── Batch size ──
    per_device_train_batch_size=4,         # Process 4 examples at a time on GPU
    per_device_eval_batch_size=4,          # Same for validation
    gradient_accumulation_steps=4,         # Accumulate gradients over 4 steps
                                           # Effective batch size = 4 × 4 = 16
                                           # Bigger effective batch = more stable training

    # ── Learning rate ──
    learning_rate=2e-4,                    # How fast the model updates weights
                                           # 2e-4 (0.0002) is standard for QLoRA
                                           # Too high = chaotic learning, too low = no learning

    warmup_steps=10,                       # Gradually increase learning rate for first 10 steps
                                           # Prevents the model from making wild changes at the start

    # ── Memory optimization ──
    fp16=True,                             # Use 16-bit floating point (half precision)
                                           # Cuts memory usage in half, almost no quality loss

    optim="paged_adamw_8bit",              # Memory-efficient optimizer
                                           # AdamW is the standard optimizer for transformers
                                           # "paged_8bit" version uses much less memory

    # ── Logging & saving ──
    logging_steps=10,                      # Print training loss every 10 steps
    eval_strategy="steps",                 # Run validation at regular intervals
    eval_steps=50,                         # Validate every 50 steps
    save_strategy="steps",                 # Save model checkpoint at regular intervals
    save_steps=50,                         # Save every 50 steps
    save_total_limit=2,                    # Keep only the 2 most recent checkpoints (saves disk)

    load_best_model_at_end=True,           # After training, load the checkpoint with best val loss
    metric_for_best_model="eval_loss",     # Use validation loss to determine "best"

    # ── Misc ──
    report_to="none",                      # Don't send logs to external services
    seed=42,                               # Random seed for reproducibility
)

print("✅ Training arguments configured!")
print(f"   Epochs: {training_args.num_train_epochs}")
print(f"   Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"   Learning rate: {training_args.learning_rate}")


✅ Training arguments configured!
   Epochs: 3
   Effective batch size: 16
   Learning rate: 0.0002


In [15]:
# Cell 7: Create trainer and START TRAINING! 🚀

def formatting_func(example):
    return example["text"]

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    processing_class=tokenizer, # Changed back to processing_class as 'tokenizer' was an unexpected keyword
    # text_field="text",             # This caused the error, removed it
    formatting_func=formatting_func, # Use formatting_func instead to specify text column
    # max_seq_length=512,                    # Maximum token length per example - Removed, as it caused a TypeError
                                           # Truncates longer texts to save memory
    # packing=False,                         # Don't pack multiple examples into one sequence - Removed, as it caused a TypeError
)

print("🚀 Starting training...")
print("   This will take approximately 20-40 minutes on T4 GPU.")
print("   You'll see the loss decrease over time — that means the model is learning!")
print()

# TRAIN!
train_result = trainer.train()

# Print final results
print("\n" + "=" * 60)
print("✅ TRAINING COMPLETE!")
print("=" * 60)
print(f"   Total training time: {train_result.metrics['train_runtime']:.0f} seconds")
print(f"   Final training loss: {train_result.metrics['train_loss']:.4f}")

Applying formatting function to train dataset:   0%|          | 0/897 [00:00<?, ? examples/s]

Converting train dataset to ChatML:   0%|          | 0/897 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/897 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/897 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/897 [00:00<?, ? examples/s]

...

🚀 Starting training...
   This will take approximately 20-40 minutes on T4 GPU.
   You'll see the loss decrease over time — that means the model is learning!



/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,1.164700,1.347253
100,0.670100,1.416406
150,0.402400,1.535576


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt


✅ TRAINING COMPLETE!
   Total training time: 2710 seconds
   Final training loss: 0.8560


In [16]:
# Cell 8: Save the fine-tuned LoRA adapter

# Save only the LoRA adapter weights (not the full 7B model)
# This will be a small file (~80-160MB) instead of ~4GB
SAVE_PATH = "./financial-sentiment-adapter"

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

# List the saved files
import os
print("📁 Saved files:")
for f in os.listdir(SAVE_PATH):
    size_mb = os.path.getsize(os.path.join(SAVE_PATH, f)) / 1e6
    print(f"   {f} ({size_mb:.1f} MB)")

print(f"\n✅ Model saved to '{SAVE_PATH}'")


📁 Saved files:
   special_tokens_map.json (0.0 MB)
   training_args.bin (0.0 MB)
   adapter_config.json (0.0 MB)
   vocab.json (2.8 MB)
   merges.txt (1.7 MB)
   chat_template.jinja (0.0 MB)
   README.md (0.0 MB)
   tokenizer_config.json (0.0 MB)
   added_tokens.json (0.0 MB)
   adapter_model.safetensors (161.5 MB)
   tokenizer.json (11.4 MB)

✅ Model saved to './financial-sentiment-adapter'


In [17]:
# Cell 9: Zip and download the adapter

import shutil

# Zip the adapter folder
shutil.make_archive("financial-sentiment-adapter", "zip", SAVE_PATH)

# Download to your Mac
from google.colab import files
files.download("financial-sentiment-adapter.zip")

print("✅ Download started! Check your browser's download folder.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download started! Check your browser's download folder.


In [18]:
# Cell 10: Test the fine-tuned model on unseen data

import re

# The instruction we used during training (must be EXACTLY the same)
INSTRUCTION = (
    "You are a financial sentiment analyst. "
    "Classify the sentiment of the following earnings call transcript segment. "
    "Respond with exactly one word: positive, negative, or neutral."
)

def predict_sentiment(transcript_text):
    """
    Given a transcript segment, use our fine-tuned model to predict sentiment.
    Returns the predicted label: 'positive', 'negative', or 'neutral'
    """
    # Format the input exactly like training data (but WITHOUT the response)
    prompt = (
        f"### Instruction:\n{INSTRUCTION}\n\n"
        f"### Input:\n{transcript_text}\n\n"
        f"### Response:\n"
    )

    # Convert text to tokens
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate a response (the model completes the text after "### Response:\n")
    with torch.no_grad():  # Don't compute gradients (saves memory, faster)
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,       # We only need 1 word, but allow 10 for safety
            temperature=0.1,         # Low temperature = more deterministic/confident
            do_sample=False,         # Greedy decoding — pick the most likely token
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the NEW tokens (not the input prompt)
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip().lower()

    # Extract just the sentiment word
    # The model might output extra text, so we look for our target words
    for label in ["positive", "negative", "neutral"]:
        if label in response:
            return label

    # If no valid label found, return the raw response for debugging
    return f"UNKNOWN({response})"

# ─── Test on a few examples first ────────────────────────────────

print("🧪 Testing on 5 sample examples first...\n")

for i in range(5):
    example = dataset["test"][i]
    actual = example["output"]  # True label
    transcript = example["input"][:150]  # First 150 chars for display

    predicted = predict_sentiment(example["input"])

    match = "✅" if predicted == actual else "❌"
    print(f"  {match} Example {i+1}")
    print(f"     Text: {transcript}...")
    print(f"     Actual: {actual}")
    print(f"     Predicted: {predicted}")
    print()


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


🧪 Testing on 5 sample examples first...



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  ✅ Example 1
     Text: So we produce one simple form for you at the end of the year and government's copy, you get a copy, just like any other kind of financial service firm...
     Actual: neutral
     Predicted: neutral



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  ✅ Example 2
     Text: The first item of business is the election of two class II directors. The nominees are Don Hutchison and Gena Lee Man....
     Actual: neutral
     Predicted: neutral



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  ✅ Example 3
     Text: During this call, we may discuss certain non-GAAP financial measures. In our press release, slides accompanying this webcast and our filings with the ...
     Actual: neutral
     Predicted: neutral



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  ❌ Example 4
     Text: The regulators can't meet with every single company in crypto....
     Actual: neutral
     Predicted: negative

  ✅ Example 5
     Text: Greetings, and welcome to the Eyenovia Fourth Quarter and Full Year 2023 Earnings Call. [Operator Instructions]. As a reminder, this conference is bei...
     Actual: neutral
     Predicted: neutral



In [19]:
# Cell 11: Evaluate on ALL test examples + compute metrics

from tqdm import tqdm  # Progress bar

print("📊 Running predictions on full test set...")
print(f"   Total test examples: {len(dataset['test'])}\n")

actual_labels = []
predicted_labels = []

for i in tqdm(range(len(dataset["test"])), desc="Predicting"):
    example = dataset["test"][i]
    actual = example["output"]
    predicted = predict_sentiment(example["input"])

    actual_labels.append(actual)
    predicted_labels.append(predicted)

print(f"\n✅ All {len(actual_labels)} predictions complete!")

# Show raw results
correct = sum(1 for a, p in zip(actual_labels, predicted_labels) if a == p)
total = len(actual_labels)
print(f"\n📊 Quick accuracy: {correct}/{total} = {100 * correct / total:.1f}%")


📊 Running predictions on full test set...
   Total test examples: 106



Predicting: 100%|██████████| 106/106 [02:04<00:00,  1.17s/it]


✅ All 106 predictions complete!

📊 Quick accuracy: 82/106 = 77.4%


In [20]:
# Cell 12: Detailed evaluation metrics

from sklearn.metrics import classification_report, confusion_matrix

# ─── Classification Report ────────────────────────────────────────
# Shows precision, recall, F1 for each class

print("=" * 60)
print("📊 DETAILED CLASSIFICATION REPORT")
print("=" * 60)

# Filter out any UNKNOWN predictions for clean metrics
valid_indices = [i for i, p in enumerate(predicted_labels) if not p.startswith("UNKNOWN")]
valid_actual = [actual_labels[i] for i in valid_indices]
valid_predicted = [predicted_labels[i] for i in valid_indices]

invalid_count = len(actual_labels) - len(valid_indices)
if invalid_count > 0:
    print(f"\n⚠️  {invalid_count} predictions were invalid (model gave unexpected output)")

print()
print(classification_report(
    valid_actual,
    valid_predicted,
    labels=["positive", "negative", "neutral"],
    digits=3
))

# ─── Confusion Matrix ─────────────────────────────────────────────
# Shows what the model predicted vs what was actually correct

print("=" * 60)
print("📊 CONFUSION MATRIX")
print("=" * 60)
print("(Rows = Actual, Columns = Predicted)\n")

labels = ["positive", "negative", "neutral"]
cm = confusion_matrix(valid_actual, valid_predicted, labels=labels)

# Pretty print
print(f"{'':>12} {'positive':>10} {'negative':>10} {'neutral':>10}")
print("─" * 45)
for i, label in enumerate(labels):
    print(f"{label:>12} {cm[i][0]:>10} {cm[i][1]:>10} {cm[i][2]:>10}")

# ─── What the metrics mean ─────────────────────────────────────────
print("\n" + "=" * 60)
print("📖 WHAT THESE NUMBERS MEAN")
print("=" * 60)
print("""
  Precision: Of all times the model SAID 'positive', how often was it right?
  Recall:    Of all ACTUAL 'positive' examples, how many did the model catch?
  F1-score:  Balance between precision and recall (harmonic mean)
  Support:   How many test examples exist for each class
""")


📊 DETAILED CLASSIFICATION REPORT

              precision    recall  f1-score   support

    positive      1.000     0.484     0.652        31
    negative      0.625     0.500     0.556        10
     neutral      0.747     0.954     0.838        65

    accuracy                          0.774       106
   macro avg      0.791     0.646     0.682       106
weighted avg      0.809     0.774     0.757       106

📊 CONFUSION MATRIX
(Rows = Actual, Columns = Predicted)

               positive   negative    neutral
─────────────────────────────────────────────
    positive         15          0         16
    negative          0          5          5
     neutral          0          3         62

📖 WHAT THESE NUMBERS MEAN

  Precision: Of all times the model SAID 'positive', how often was it right?
  Recall:    Of all ACTUAL 'positive' examples, how many did the model catch?
  F1-score:  Balance between precision and recall (harmonic mean)
  Support:   How many test examples exist for eac

In [21]:
# Cell 13: Save results to a file

import json

results = {
    "model": "Qwen/Qwen2.5-7B + QLoRA",
    "test_size": len(actual_labels),
    "accuracy": round(100 * correct / total, 2),
    "valid_predictions": len(valid_indices),
    "invalid_predictions": invalid_count,
    "classification_report": classification_report(
        valid_actual, valid_predicted,
        labels=["positive", "negative", "neutral"],
        output_dict=True
    ),
    "predictions": [
        {"actual": a, "predicted": p}
        for a, p in zip(actual_labels, predicted_labels)
    ]
}

with open("evaluation_results.json", "w") as f:
    json.dump(results, f, indent=2)

# Download to Mac
from google.colab import files
files.download("evaluation_results.json")

print("✅ Results saved and downloading!")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Results saved and downloading!
